## CNN-LSTM con una oportunidad justa
Tres experimentos, siempre evaluados sobre ventanas **reales y sin modificar**:
1. **Conjunto**: BIDMC + CapnoBase, 5 particiones por sujeto; error reportado por base.
2. **Cruce**: BIDMC → CapnoBase y CapnoBase → BIDMC.
3. Cada uno **con y sin aumento** por estiramiento temporal (solo en entrenamiento).

La FFT sigue congelada (`results/config_congelada.json`) y se muestra al lado como referencia.

In [ ]:
# j-1 — Imports, configuración congelada y parámetros de los experimentos
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb
from scipy.signal import butter, decimate, detrend, filtfilt, resample, resample_poly
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MIN, RPM_MAX = CFG["rpm_min"], CFG["rpm_max"]
N = VENTANA_S * FS

BIDMC = [f"bidmc{i:02d}" for i in range(1, 54)]
CAPNO_DIR = BASE_DIR / "data" / "capnobase" / "csv"
CAPNO = sorted(p.name.replace("_signal.csv", "") for p in CAPNO_DIR.glob("*_signal.csv"))

K = 5                    # particiones por sujeto (experimento conjunto)
EPOCAS = 60
R_MIN, R_MAX = 0.75, 1.35  # factor de estiramiento: RR_nueva = RR * r
COPIAS_AUG = 2           # copias aumentadas por ventana de entrenamiento
HR_MIN, HR_MAX = 40, 180  # se descartan copias con frecuencia cardiaca no fisiológica
print(len(BIDMC), "BIDMC |", len(CAPNO), "CapnoBase")


In [ ]:
# j-2 — Lectura de registros (señal a 125 Hz + marcas de respiración)
def leer_bidmc(p):
    ruta = str(BASE_DIR / "data" / p)
    rec = wfdb.rdrecord(ruta)
    ppg = rec.p_signal[:, [s.strip(" ,").upper() for s in rec.sig_name].index("PLETH")]
    ann = wfdb.rdann(ruta, "breath")
    aux = np.array([(a or "").strip(" \x00") for a in ann.aux_note])
    fa = ann.fs or rec.fs
    return ppg, {k: np.sort(ann.sample[aux == k]) / fa for k in ("ann1", "ann2")}

def _lista(c):
    return np.array([]) if pd.isna(c) or str(c).strip() == "" else np.array(str(c).split(), dtype=float)

def leer_capno(c):
    sig = pd.read_csv(CAPNO_DIR / f"{c}_signal.csv")
    par = pd.read_csv(CAPNO_DIR / f"{c}_param.csv").iloc[0]
    lab = pd.read_csv(CAPNO_DIR / f"{c}_labels.csv", keep_default_na=False).iloc[0]
    ppg = resample_poly(sig["pleth_y"].to_numpy(float), FS, int(par["samplingrate_pleth"]))
    return ppg, {"ann1": np.sort(_lista(lab["co2_startexp_x"])) / int(par["samplingrate_co2"])}

REG = {}
for p in BIDMC:
    REG[("BIDMC", p)] = leer_bidmc(p)
for c in CAPNO:
    REG[("CapnoBase", c)] = leer_capno(c)


In [ ]:
# j-3 — Ventanas, referencia, validez y FFT congelada (mismo criterio que 02 y 03)
def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def rr_fft(seg, fs=FS, nfft=2048):
    q = int(fs // 5)
    x = decimate(decimate(seg, 5, zero_phase=True), q // 5, zero_phase=True)
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, q / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    banda = (f >= RPM_MIN) & (f <= RPM_MAX)
    return f[np.argmax(np.where(banda, P, -np.inf))]

def fc_ventana(w, fs=FS):
    """Frecuencia cardiaca aproximada (lpm): pico espectral en 0.7–3 Hz."""
    f = np.fft.rfftfreq(8192, 1 / fs)
    P = np.abs(np.fft.rfft(detrend(w) * np.hanning(len(w)), 8192))
    b = (f >= 0.7) & (f <= 3.0)
    return 60 * f[b][np.argmax(P[b])]

filas = []
for (base, rid), (ppg, marcas) in REG.items():
    bl, al = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv_fft = filtfilt(bl, al, ppg)
    bl, al = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv = filtfilt(bl, al, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv[i:i + n5]) for i in range(0, len(riiv) - n5 + 1, n5)])
    for t0 in np.round(np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S), 3):
        i0 = int(round(t0 * FS))
        exc = max(np.std(riiv[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico
        r1 = rr_ref(marcas["ann1"], t0, t0 + VENTANA_S)
        fiable = True if "ann2" not in marcas else abs(r1 - rr_ref(marcas["ann2"], t0, t0 + VENTANA_S)) <= CFG["umbral_anot"]
        filas.append({"Base": base, "Paciente": rid, "Inicio (s)": t0, "RR referencia (rpm)": r1,
                      "RR FFT (rpm)": rr_fft(riiv_fft[i0:i0 + N]),
                      "Válida": bool(exc <= CFG["umbral_art"] and np.isfinite(r1) and fiable),
                      "FC (lpm)": fc_ventana(ppg[i0:i0 + N])})
D = pd.DataFrame(filas)
print(D.groupby("Base")["Válida"].agg(["size", "sum"]))

def ventana(base, rid, t0, r=1.0):
    """Ventana z-score de 30 s. Con r != 1 toma 30·r s de señal real y la remuestrea a 30 s."""
    ppg = REG[(base, rid)][0]
    i0 = int(round(t0 * FS))
    seg = ppg[i0:i0 + int(round(N * r))]
    if r != 1.0:
        seg = resample(seg, N)
    return ((seg - seg.mean()) / (seg.std() + 1e-6)).astype(np.float32)

X = np.stack([ventana(b, p, t) for b, p, t in zip(D["Base"], D["Paciente"], D["Inicio (s)"])])
y = D["RR referencia (rpm)"].to_numpy(np.float32)
print("X:", X.shape)


In [ ]:
# j-4 — Aumento por estiramiento temporal (solo para ventanas de entrenamiento)
_valida = {(b, p, t): v for b, p, t, v in zip(D["Base"], D["Paciente"], D["Inicio (s)"], D["Válida"])}

def aumentar(idx, rng):
    """Copias estiradas de las ventanas idx. La etiqueta se recalcula con las marcas reales
    del tramo usado (30·r s) y se escala por r. Se descartan copias que:
    salen del registro, tocan una ventana no válida, o dan RR/FC fuera de rango fisiológico."""
    Xa, ya = [], []
    for i in idx:
        b, p, t0 = D.at[i, "Base"], D.at[i, "Paciente"], D.at[i, "Inicio (s)"]
        ppg, marcas = REG[(b, p)]
        for _ in range(COPIAS_AUG):
            r = rng.uniform(R_MIN, R_MAX)
            t1 = t0 + VENTANA_S * r
            if t1 * FS > len(ppg):
                continue
            if r > 1 and not _valida.get((b, p, round(t0 + PASO_S, 3)), False):
                continue                       # el tramo extra debe estar también limpio
            rr = rr_ref(marcas["ann1"], t0, t1)
            if not np.isfinite(rr):
                continue
            rr_nueva, fc_nueva = rr * r, D.at[i, "FC (lpm)"] * r
            if not (RPM_MIN <= rr_nueva <= RPM_MAX and HR_MIN <= fc_nueva <= HR_MAX):
                continue
            Xa.append(ventana(b, p, t0, r))
            ya.append(rr_nueva)
    return np.array(Xa, np.float32).reshape(-1, N), np.array(ya, np.float32)


In [ ]:
# j-5 — Modelo (tu arquitectura) y función de entrenamiento
def crear_modelo():
    model = keras.Sequential([
        layers.Input(shape=(N, 1)),
        layers.Conv1D(16, 15, strides=3, padding="same", activation="relu"),
        layers.MaxPooling1D(3),
        layers.Conv1D(32, 9, padding="same", activation="relu"),
        layers.MaxPooling1D(3),
        layers.LSTM(16),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)])
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])
    return model

def entrenar_y_predecir(m_train, m_test, aumento, semilla):
    """m_train / m_test: máscaras booleanas sobre D. Valida (early stopping) con un 15 % de los
    sujetos de entrenamiento. Devuelve predicciones para todas las ventanas de m_test."""
    rng = np.random.default_rng(semilla)
    m_train = m_train & D["Válida"].to_numpy()
    suj = D.loc[m_train, "Paciente"].unique()
    s_val = set(rng.choice(suj, max(1, int(round(0.15 * len(suj)))), replace=False))
    es_val = D["Paciente"].isin(s_val).to_numpy()
    tr, va = np.where(m_train & ~es_val)[0], np.where(m_train & es_val)[0]
    Xtr, ytr = X[tr], y[tr]
    n_aug = 0
    if aumento:
        Xa, ya = aumentar(tr, rng)
        Xtr, ytr, n_aug = np.concatenate([Xtr, Xa]), np.concatenate([ytr, ya]), len(ya)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(int(semilla))
    m = crear_modelo()
    m.fit(Xtr[:, :, None], ytr / 60, validation_data=(X[va, :, None], y[va] / 60),
          epochs=EPOCAS, batch_size=32, verbose=0, shuffle=True,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    print(f"    train={len(tr)} (+{n_aug} aumentadas)  val={len(va)}  test={m_test.sum()}")
    return m.predict(X[m_test, :, None], verbose=0).ravel() * 60


In [ ]:
# j-6 — Experimentos
es_bidmc = (D["Base"] == "BIDMC").to_numpy()
es_capno = ~es_bidmc
suj = np.array(sorted(D["Paciente"].unique()))
grupo = dict(zip(np.random.default_rng(0).permutation(suj), np.arange(len(suj)) % K))
D["Partición"] = D["Paciente"].map(grupo)

for aumento in (False, True):
    tag = "aug" if aumento else "sin_aug"
    print(f"== Conjunto, {tag}")
    col = f"CNN conjunto {tag}"
    D[col] = np.nan
    for k in range(K):
        te = (D["Partición"] == k).to_numpy()
        D.loc[te, col] = entrenar_y_predecir(~te, te, aumento, semilla=100 + k)
    print(f"== BIDMC → CapnoBase, {tag}")
    D[f"CNN B→C {tag}"] = np.nan
    D.loc[es_capno, f"CNN B→C {tag}"] = entrenar_y_predecir(es_bidmc, es_capno, aumento, semilla=200)
    print(f"== CapnoBase → BIDMC, {tag}")
    D[f"CNN C→B {tag}"] = np.nan
    D.loc[es_bidmc, f"CNN C→B {tag}"] = entrenar_y_predecir(es_capno, es_bidmc, aumento, semilla=300)

D.to_csv(OUT / "cnn_justa_ventanas.csv", index=False)


In [ ]:
# j-7 — Resultados por base (MAE mediano por sujeto, solo ventanas válidas)
V = D[D["Válida"]].copy()
cols = {"FFT (congelada)": "RR FFT (rpm)",
        "CNN conjunto": "CNN conjunto sin_aug", "CNN conjunto + aug": "CNN conjunto aug",
        "CNN cruce": None, "CNN cruce + aug": None}
filas = []
for base, g in V.groupby("Base"):
    cruce = "CNN B→C" if base == "CapnoBase" else "CNN C→B"
    cols["CNN cruce"], cols["CNN cruce + aug"] = f"{cruce} sin_aug", f"{cruce} aug"
    fila = {"Base": base, "Sujetos": g["Paciente"].nunique()}
    for nombre, c in cols.items():
        e = (g[c] - g["RR referencia (rpm)"]).abs()
        por_suj = e.groupby(g["Paciente"]).mean()
        fila[nombre] = f"{por_suj.median():.2f} ({100 * (e <= 2).mean():.0f} %)"
    filas.append(fila)
tabla = pd.DataFrame(filas)
tabla.to_csv(OUT / "cnn_justa_resumen.csv", index=False)
print("MAE mediano por sujeto (rpm) y, entre paréntesis, % de ventanas con |error| ≤ 2 rpm")
display(tabla)

print("\nDiagnóstico: media de las predicciones frente a la media de la referencia, por base")
display(V.groupby("Base")[["RR referencia (rpm)", "RR FFT (rpm)", "CNN conjunto aug",
                           "CNN B→C aug", "CNN C→B aug"]].mean().round(1))
